# Lab 17.2: Medical Features in GPT-2 Small

Chapter 17, Features, Superposition and Sparse Autoencoders. Run-only notebook for the language model environment (`env/requirements-llm.txt`, kernel `xai-llm-venv`).

We load GPT-2 small with TransformerLens and a published sparse autoencoder (SAE) for the residual stream before layer 8 (24,576 latents, trained by Joseph Bloom on OpenWebText, MIT license, Hugging Face repository `jbloom/GPT2-Small-SAEs-Reformatted`). The first run downloads about 75 MB for the SAE (and about 500 MB for GPT-2 if it is not cached). Everything runs on a CPU.

In [1]:
import torch, numpy as np, time
torch.set_num_threads(2); torch.manual_seed(0)
from transformer_lens import HookedTransformer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
T0 = time.time()
model = HookedTransformer.from_pretrained("gpt2", device="cpu"); model.eval()
REPO, HOOK = "jbloom/GPT2-Small-SAEs-Reformatted", "blocks.8.hook_resid_pre"
w = load_file(hf_hub_download(REPO, f"{HOOK}/sae_weights.safetensors"))
W_enc, b_enc, W_dec, b_dec = w["W_enc"], w["b_enc"], w["W_dec"], w["b_dec"]
encode = lambda x: torch.relu((x - b_dec) @ W_enc + b_enc)
decode = lambda f: f @ W_dec + b_dec
print("SAE:", tuple(W_enc.shape), "decoder row norms about", round(float(W_dec.norm(dim=1).mean()), 3))

Loaded pretrained model gpt2 into HookedTransformer


SAE: (768, 24576) decoder row norms about 1.0


## Part A. A small corpus and the SAE's quality

A few sentences in four topics. Real studies use millions of tokens; this corpus is small so that it runs in seconds, which also limits what we can conclude.

In [2]:
corpus = {
 "medicine": ["The patient was diagnosed with type 2 diabetes and started on insulin.",
  "Her blood pressure was high, so the doctor prescribed a beta blocker.",
  "The nurse checked his temperature and gave him antibiotics for the infection.",
  "Chest pain radiating to the left arm can be a sign of a heart attack.",
  "The surgeon removed the tumor and the biopsy came back benign.",
  "Pregnant women are screened for preeclampsia during prenatal visits.",
  "The vaccine reduced hospital admissions for influenza among older adults.",
  "Symptoms of asthma include wheezing, coughing and shortness of breath."],
 "sports": ["The striker scored twice in the second half to win the match.",
  "She won the gold medal in the hundred meter final.",
  "The coach changed the lineup before the playoff game.",
  "He hit a home run in the bottom of the ninth inning.",
  "The tennis match went to a fifth set tiebreak.",
  "Fans filled the stadium for the championship final."],
 "code": ["The function returns None if the list is empty.",
  "Use a for loop to iterate over the dictionary keys.",
  "The compiler threw a syntax error on line twelve.",
  "Import numpy and create an array of zeros.",
  "The server crashed because of a null pointer exception.",
  "Commit your changes before you merge the branch."],
 "finance": ["The bank raised interest rates by a quarter point.",
  "Shares fell after the company reported weak quarterly earnings.",
  "The loan was approved after a review of her credit history.",
  "Inflation rose faster than economists had expected.",
  "Investors moved money into government bonds.",
  "The insurer increased premiums for drivers under twenty five."],
}
rows = []
for topic, sents in corpus.items():
    for s in sents:
        toks = model.to_tokens(s)
        _, cache = model.run_with_cache(toks, names_filter=HOOK)
        x = cache[HOOK][0, 1:]                       # drop the beginning-of-text position
        rows.append((topic, s, model.to_str_tokens(toks)[1:], x, encode(x)))
X = torch.cat([r[3] for r in rows]); F = torch.cat([r[4] for r in rows])
fvu = float(((X - decode(F)) ** 2).sum() / ((X - X.mean(0)) ** 2).sum())
print(f"tokens {X.shape[0]}, L0 (active latents per token) {float((F > 0).float().sum(1).mean()):.1f}, "
      f"variance unexplained {fvu:.3f}")

tokens 301, L0 (active latents per token) 63.4, variance unexplained 0.158


In [3]:
def loss_with(hook_fn, s):
    return float(model.run_with_hooks(model.to_tokens(s), return_type="loss", fwd_hooks=[(HOOK, hook_fn)]))
def splice(act, hook):                               # replace activations by the SAE reconstruction
    a = act.clone(); a[:, 1:] = decode(encode(act[:, 1:])); return a
def mean_ablate(act, hook):                          # replace activations by their corpus mean
    a = act.clone(); a[:, 1:] = X.mean(0); return a
clean = np.mean([float(model(model.to_tokens(r[1]), return_type="loss")) for r in rows])
sae = np.mean([loss_with(splice, r[1]) for r in rows])
ablated = np.mean([loss_with(mean_ablate, r[1]) for r in rows])
print(f"next-token loss: clean {clean:.3f}, with SAE spliced in {sae:.3f}, mean-ablated {ablated:.3f}")
print(f"share of loss recovered by the SAE: {(ablated - sae) / (ablated - clean):.3f}")

next-token loss: clean 3.828, with SAE spliced in 3.951, mean-ablated 7.618
share of loss recovered by the SAE: 0.968


**Checkpoint 1.** About 300 tokens; roughly 60 latents active per token out of 24,576; about 16 percent of the variance unexplained; splicing in the reconstruction raises the loss a little (about 3.83 to 3.95) and recovers about 97 percent of the loss against mean ablation. A dictionary can recover most of the loss and still leave a sixth of the variance out.

## Part B. Which latents are selective for medical text?

For each latent: the share of medical tokens on which it is active minus the share of other tokens. Then, for the most selective latents, the tokens where they are most active, and their output effects (the decoder direction pushed through the unembedding, as in the logit lens of Chapter 15).

In [4]:
topic = np.concatenate([[r[0]] * r[3].shape[0] for r in rows])
is_med = torch.tensor(topic == "medicine")
on = (F > 0).float()
sel = on[is_med].mean(0) - on[~is_med].mean(0)
top = torch.topk(sel, 6).indices.tolist()
tokens = sum([r[2] for r in rows], [])
for i in top:
    best = torch.topk(F[:, i], 5).indices.tolist()
    logit = torch.topk(W_dec[i] @ model.W_U, 6).indices.tolist()
    print(f"latent {i:5d}: active on {on[is_med][:, i].mean():.2f} of medical tokens, "
          f"{on[~is_med][:, i].mean():.2f} of others")
    print("   top tokens:", [(tokens[j], round(float(F[j, i]), 1)) for j in best])
    print("   pushes up :", [model.to_string(t) for t in logit])
    print(f"   compare: https://www.neuronpedia.org/gpt2-small/8-res-jb/{i}")

latent  9151: active on 0.58 of medical tokens, 0.01 of others
   top tokens: [('.', 5.6), ('.', 5.1), (' during', 5.1), ('.', 4.8), (' started', 4.3)]
   pushes up : [' clinical', ' dietary', ' pediatric', ' hormonal', ' patients', ' Pediatrics']
   compare: https://www.neuronpedia.org/gpt2-small/8-res-jb/9151
latent   357: active on 0.54 of medical tokens, 0.01 of others
   top tokens: [(' reduced', 7.7), (' for', 6.0), (' among', 5.6), (' type', 5.6), (' of', 5.3)]
   pushes up : [' biomark', ' cannabinoid', ' genome', ' metabolic', ' lipid', ' epigen']
   compare: https://www.neuronpedia.org/gpt2-small/8-res-jb/357
latent 14773: active on 0.42 of medical tokens, 0.01 of others
   top tokens: [(' and', 6.8), ('.', 6.8), ('.', 6.7), (' the', 6.2), ('.', 5.8)]
   pushes up : [' surgery', ' amput', ' surgeries', 'Doctors', ' tendon', ' medically']
   compare: https://www.neuronpedia.org/gpt2-small/8-res-jb/14773
latent  6435: active on 0.39 of medical tokens, 0.01 of others
   top toke

**Checkpoint 2.** The most selective latents (in our run: 9151, 357, 14773, 6435, 9818, 23479) are almost never active on the other topics. Several fire on ordinary words such as "the", "and" or "." *inside* medical sentences: they track the context, not the word. Their output effects are medical (for example "clinical", "patients"; "pneumonia", "tuberculosis"; "symptoms", "nausea"). Open the Neuronpedia links to compare the published automatic labels with what you see here.

## Part C. Are these features necessary? Ablation

A prompt where GPT-2 should continue with a medical test. We remove the most selective latents (subtract their contribution from the residual stream at every position) and compare the next-token distribution.

In [5]:
prompt = "The patient was admitted to the hospital with chest pain, and the doctor ordered an"
ptoks = model.to_tokens(prompt)
def next_probs(hooks=()):
    return torch.softmax(model.run_with_hooks(ptoks, fwd_hooks=list(hooks))[0, -1], -1)
def ablate(ids):
    def hook(act, hook):
        f = encode(act[0]); a = act.clone()
        for i in ids:
            a[0] -= f[:, i:i + 1] * W_dec[i]
        return a
    return hook
show = lambda p: [(model.to_string(int(t)), round(float(v), 3)) for v, t in zip(*torch.topk(p, 6))]
p0 = next_probs()
print("clean        :", show(p0))
for k in (1, 3, 6):
    p1 = next_probs([(HOOK, ablate(top[:k]))])
    tv = 0.5 * float((p1 - p0).abs().sum())
    print(f"ablate top {k}:", show(p1), f" total variation {tv:.3f}")

clean        : [(' MRI', 0.16), (' emergency', 0.083), (' ultrasound', 0.069), (' IV', 0.048), (' X', 0.046), (' intraven', 0.037)]


ablate top 1: [(' MRI', 0.166), (' emergency', 0.084), (' ultrasound', 0.071), (' X', 0.047), (' IV', 0.044), (' urgent', 0.032)]  total variation 0.031
ablate top 3: [(' MRI', 0.156), (' emergency', 0.088), (' ultrasound', 0.074), (' X', 0.045), (' IV', 0.039), (' ambulance', 0.036)]  total variation 0.052


ablate top 6: [(' MRI', 0.157), (' ultrasound', 0.083), (' emergency', 0.079), (' X', 0.043), (' IV', 0.034), (' ambulance', 0.034)]  total variation 0.080


**Checkpoint 3.** The top predictions stay medical ("MRI", "emergency", "ultrasound") and their probabilities barely move (total variation of a few hundredths). The selective latents are not *necessary* for this prediction: the information is also carried elsewhere, by other latents, by the reconstruction error and by later layers reading earlier positions.

## Part D. Are they sufficient? Steering with a control

Now we add one latent's decoder direction to the residual stream of an unrelated prompt and generate greedily. As a control we add a random direction of the same length. Try other latents and strengths in the **EDIT** cell.

In [6]:
# EDIT: latent and strength
LATENT, STRENGTH = 9151, 40
neutral = model.to_tokens("On Saturday morning I went to the")
def generate(hooks, n=14):
    t = neutral.clone()
    for _ in range(n):
        t = torch.cat([t, model.run_with_hooks(t, fwd_hooks=list(hooks))[0, -1].argmax().view(1, 1)], 1)
    return model.to_string(t[0, neutral.shape[1]:])
def add(direction):
    def hook(act, hook):
        a = act.clone(); a[0, 1:] += STRENGTH * direction; return a
    return hook
g = torch.Generator().manual_seed(0)
rand_dir = torch.randn(768, generator=g); rand_dir /= rand_dir.norm()
print("clean          :", repr(generate([])))
print(f"latent {LATENT}   :", repr(generate([(HOOK, add(W_dec[LATENT] / W_dec[LATENT].norm()))])))
print("random control :", repr(generate([(HOOK, add(rand_dir))])))
for lat in top[:4]:
    print(f"latent {lat:5d} :", repr(generate([(HOOK, add(W_dec[lat] / W_dec[lat].norm()))])))
print(f"total run time {time.time() - T0:.0f} s")

clean          : ' local grocery store to buy some groceries. I was told that the store'


latent 9151   : ' emergency room of a young woman in her 20s who had been diagnosed'


random control : ' local grocery store to buy some groceries. I was looking for a bag'


latent  9151 : ' emergency room of a young woman in her 20s who had been diagnosed'


latent   357 : ' onset of my post-traumatic stress disorder (PTSD) in'


latent 14773 : ' hospital. I was in a good place. I was in a good'


latent  6435 : ' hospital with a broken leg. I was in the hospital for a few'
total run time 25 s


**Checkpoint 4.** The clean continuation is about a grocery store. Adding a medical latent's direction turns it toward a hospital, a diagnosis or symptoms; the random direction of the same length does not. The latents are *sufficient* to push the model toward medical text, even though removing them did not change the medical prediction in Part C.

## Questions

1. Write a feature card for one latent: label, top tokens, output effects, ablation result, steering result, and one counterexample from your own test text.
2. Part C and Part D seem to disagree. Explain why both can be true, using the words *selective*, *necessary* and *sufficient*.
3. What would you need to claim that GPT-2 "uses a concept of disease" in a particular prediction? Which chapter's methods would you add (Chapter 15, 16 or 18), and why?